In [2]:
# series, Dataframes and index
import pandas as pd



data = {

    "order_id":   [101, 102, 103, 104, 105, 106],

    "branch":     ["Nairobi", "Mombasa", "Nairobi", "Kisumu", "Mombasa", "Nairobi"],

    "product":    ["Phone", "Laptop", "Headphones", "Phone", "Phone", "Laptop"],

    "quantity":   [2, 1, 5, 1, 3, 2],

    "unit_price": [15000, 65000, 2500, 15000, 15000, 65000],

}

df = pd.DataFrame(data)

print(df.head())

print(df.shape)

   order_id   branch     product  quantity  unit_price
0       101  Nairobi       Phone         2       15000
1       102  Mombasa      Laptop         1       65000
2       103  Nairobi  Headphones         5        2500
3       104   Kisumu       Phone         1       15000
4       105  Mombasa       Phone         3       15000
(6, 5)


PYTHON

sales = pd.read_csv(

    "duka_sales.csv",

    parse_dates=["order_date"],

    dtype={"order_id": "int64"},

)



# Excel files: choose the sheet by name

# budget = pd.read_excel("budget_2026.xlsx", sheet_name="Q1")

In [5]:
# The first five minutes: Inspecting a Dataset

df["revenue"] = df["quantity"] * df["unit_price"]

print(df.describe()[["quantity", "revenue"]],round(1))

       quantity        revenue
count  6.000000       6.000000
mean   2.333333   49583.333333
std    1.505545   44000.473482
min    1.000000   12500.000000
25%    1.250000   18750.000000
50%    2.000000   37500.000000
75%    2.750000   60000.000000
max    5.000000  130000.000000 1


In [6]:
# Selecting Data: Columns, loc and iloc

print(df.iloc[0:2, 0:3]) # by position
# by label + condition
print(df.loc[df["revenue"] > 40000, ["order_id", "branch", "revenue"]])

   order_id   branch product
0       101  Nairobi   Phone
1       102  Mombasa  Laptop
   order_id   branch  revenue
1       102  Mombasa    65000
4       105  Mombasa    45000
5       106  Nairobi   130000


In [7]:
# Filtering Rows

nairobi = df[df["branch"] == "Nairobi"]
big_phone_orders = df[(df["product"] == "Phone") & (df["quantity"] >= 2)]
costal_or_lake = df[df["branch"].isin(["Mombasa", "Kisumu"])]
mid_range = df[df["revenue"].between(15000, 65000)]
same_with_query = df.query("product == 'Phone' and quantity >= 2")

print(big_phone_orders[["order_id", "branch", "quantity"]])

   order_id   branch  quantity
0       101  Nairobi         2
4       105  Mombasa         3


In [8]:
# Sorting and Ranking

print(df.sort_values("revenue", ascending=False).head(3))

   order_id   branch product  quantity  unit_price  revenue
5       106  Nairobi  Laptop         2       65000   130000
1       102  Mombasa  Laptop         1       65000    65000
4       105  Mombasa   Phone         3       15000    45000


In [10]:
#Creating and Transforming Columns

import numpy as np

df["order_size"] = np.where(df["revenue"] >= 50000, "Large", "Standard")
region_map = {"Nairobi": "Central", "Mombasa": "Coast", "Kisumu": "Western"}
df["Region"] = df["branch"].map(region_map)
df["revenue_k"] = (df["revenue"] / 1000).round(1)

print(df[["order_id", "branch", "Region", "revenue_k", "order_size"]])

   order_id   branch   Region  revenue_k order_size
0       101  Nairobi  Central       30.0   Standard
1       102  Mombasa    Coast       65.0      Large
2       103  Nairobi  Central       12.5   Standard
3       104   Kisumu  Western       15.0   Standard
4       105  Mombasa    Coast       45.0   Standard
5       106  Nairobi  Central      130.0      Large


In [13]:
# Counting and Summarising

print(df["product"].value_counts())
print(df.groupby("branch")["revenue"].sum().sort_values(ascending=False))

branch_totals = df.groupby("branch")["revenue"].sum().reset_index()
print(branch_totals)

product
Phone         3
Laptop        2
Headphones    1
Name: count, dtype: int64
branch
Nairobi    172500
Mombasa    110000
Kisumu      15000
Name: revenue, dtype: int64
    branch  revenue
0   Kisumu    15000
1  Mombasa   110000
2  Nairobi   172500


#  Excel and Google Sheets for Analysts

EXCEL · FORMULAS

=SUMIFS(H:H, C:C, "Nairobi")                         // revenue for Nairobi

=COUNTIFS(C:C, "Nairobi", F:F, "Online")             // Nairobi online orders

=AVERAGEIFS(H:H, D:D, "Laptop")                      // average laptop order value

=XLOOKUP(C2, Branches!A:A, Branches!B:B, "Unknown")  // fetch region for branch in C2

=H2/SUM($H$2:$H$1001)                                // share of total

In [ ]:
# A Messy Dataset to Work With

import pandas as pd

import numpy as np

raw = pd.DataFrame({
    "txn_id": ["T01", "T02", "T02", "T03", "T04", "T05"],
    "customer": [" Achieng ", "BRIAN", "BRIAN", "wanjiru", None, "Otieno"],
    "amount": ["1,500", "2300", "2300", "abc", "800", "45000"],
    "date": ["2026-01-05", "2026-01-06", "2026-01-06","2026/01/07", "2026-01-08", "2026-01-09"],
})

In [ ]:
# Dulpicates

print("Duplicate rows:", raw.duplicated().sum())
clean = raw.drop_duplicates().copy()
print(clean.shape)

In [ ]:
# Cleaning Text

clean["customer"] = clean["customer"].str.strip().str.title()
print(clean["customer"].tolist())

In [ ]:
#  Fixing Data Types

clean["amount"] = pd.to_numeric(clean["amount"].str.replace(",", ""), errors="coerce")

clean["date"] = pd.to_datetime(clean["date"].str.replace("/", "-"))

print(clean.dtypes)


In [ ]:
# Missing values

print(clean.isna().sum())

clean["customer"] = clean["customer"].fillna("Unknown")

clean["amount_imputed"] = clean["amount"].isna()          # flag first

clean["amount"] = clean["amount"].fillna(clean["amount"].median())

In [ ]:
# Outliers

# Calculated on the amount column before imputation

q1, q3 = 1325.0, 12975.0

iqr = q3 - q1

upper = q3 + 1.5 * iqr

print(f"IQR = {iqr}, upper fence = {upper}")

print(clean[clean["amount"] > upper][["txn_id", "customer", "amount"]])

In [ ]:
# Working with dates

clean["day_name"] = clean["date"].dt.day_name()

print(clean[["txn_id", "date", "day_name", "amount"]])

In [ ]:
# Grouping and Aggregating

# df is the six-order Duka sample from Week 3 (with the revenue column)

summary = df.groupby("branch").agg(

  total_revenue=("revenue", "sum"),

  orders=("order_id", "count"),

  avg_order=("revenue", "mean"),

).sort_values("total_revenue", ascending=False)

print(summary)

In [ ]:
# Combining Datasets: merge and concat

customers = pd.DataFrame({"customer_id": [1, 2, 3],
                          "name": ["Achieng", "Brian", "Wanjiru"],
                          "county": ["Kisumu", "Nairobi", "Nyeri"]})
orders = pd.DataFrame({"order_id": [501, 502, 503, 504],
                       "customer_id": [1, 2, 2, 4],
                       "amount": [1200, 3400, 560, 990]})
merged = orders.merge(customers, on="customer_id", how="left")

print(merged)

In [ ]:
# Reshaping: pivot_table and melt

pivot = pd.pivot_table(df, index="branch", columns="product",
                       values="revenue", aggfunc="sum", fill_value=0)

print(pivot)


long = pivot.reset_index().melt(id_vars="branch", var_name="product",
                                value_name="revenue")
print(long.head(4))

In [ ]:
# First Steps in SQL: Joins Side by Side with pandas

import sqlite3



conn = sqlite3.connect(":memory:")   # a temporary in-memory database

orders.to_sql("orders", conn, index=False)

customers.to_sql("customers", conn, index=False)



query = """

SELECT o.order_id, o.amount, c.name, c.county

FROM orders o

LEFT JOIN customers c ON o.customer_id = c.customer_id

ORDER BY o.order_id;

"""

print(pd.read_sql(query, conn))



SELECT c.county, SUM(o.amount) AS total

FROM orders o

JOIN customers c ON o.customer_id = c.customer_id

GROUP BY c.county

ORDER BY total DESC;

In [18]:
# Data Ethics, Privacy and the Kenya Data Protection Act 2019

import hashlib



customers = pd.DataFrame({

  "name": ["Achieng Otieno", "Brian Kamau"],

  "phone": ["0712345678", "0722000111"],

  "county": ["Kisumu", "Nairobi"],

  "spend": [4500, 12000],

})



def pseudonymise(value, salt="plp-2026"):

  """Return a short, irreversible key for an identifier."""

  return hashlib.sha256((salt + value).encode()).hexdigest()[:10]



safe = customers.assign(

  customer_key=customers["phone"].apply(pseudonymise),

  phone_masked=customers["phone"].str[:4] + "****" + customers["phone"].str[-2:],

).drop(columns=["name", "phone"])

print(safe)

    county  spend customer_key phone_masked
0   Kisumu   4500   2bb5a05279   0712****78
1  Nairobi  12000   7f11fc89f6   0722****11
